# Fine-tune LLM using LoRA and QLoRA

In [ ]:
!git clone https://github.com/shosakaue0808/LoRA-QLoRA-studies.git

In [ ]:
!pip install transformers peft bitsandbytes python-dotenv datasets

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
from huggingface_hub import login
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dotenv import load_dotenv
import os
# my files
from src.data_prep import GSMDataset
from src.train_utils import load_checkpoint, train
from src.lora_layer import attach_Lora_to_Linear



## login Hugging face

In [2]:
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [23]:
# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# Load Model

# LoRA base

In [3]:
model_id = "meta-llama/Llama-3.2-1B"

In [28]:


tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model_lora = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
base_model_lora.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

# QLoRA base

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model_qlora = LlamaForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
base_model_qlora.model_name = model_id
base_tokenizer_qlora = AutoTokenizer.from_pretrained(model_id)
base_tokenizer_qlora.pad_token = base_tokenizer_qlora.eos_token
base_model_qlora.config.pad_token_id = base_tokenizer_qlora.pad_token_id

In [5]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

model.embed_tokens.weight torch.Size([128256, 2048])
model.layers.0.self_attn.q_proj.weight torch.Size([2048, 2048])
model.layers.0.self_attn.k_proj.weight torch.Size([512, 2048])
model.layers.0.self_attn.v_proj.weight torch.Size([512, 2048])
model.layers.0.self_attn.o_proj.weight torch.Size([2048, 2048])
model.layers.0.mlp.gate_proj.weight torch.Size([8192, 2048])
model.layers.0.mlp.up_proj.weight torch.Size([8192, 2048])
model.layers.0.mlp.down_proj.weight torch.Size([2048, 8192])
model.layers.0.input_layernorm.weight torch.Size([2048])
model.layers.0.post_attention_layernorm.weight torch.Size([2048])
model.layers.1.self_attn.q_proj.weight torch.Size([2048, 2048])
model.layers.1.self_attn.k_proj.weight torch.Size([512, 2048])
model.layers.1.self_attn.v_proj.weight torch.Size([512, 2048])
model.layers.1.self_attn.o_proj.weight torch.Size([2048, 2048])
model.layers.1.mlp.gate_proj.weight torch.Size([8192, 2048])
model.layers.1.mlp.up_proj.weight torch.Size([8192, 2048])
model.layers.1.

In [6]:
#Set all weights freeze
for param in base_model_lora.parameters():
    param.requires_grad = False

# Attach LoRA layer to all linear layers in base model

In [7]:
print(base_model_lora.model.layers)


ModuleList(
  (0-15): 16 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
      (k_proj): Linear(in_features=2048, out_features=512, bias=False)
      (v_proj): Linear(in_features=2048, out_features=512, bias=False)
      (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
      (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
      (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  )
)


In [8]:
attach_Lora_to_Linear(base_model_lora, rank=4, alpha=16)

In [9]:
print(base_model_lora)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=2048, bias=False)
          )
          (k_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=512, bias=False)
          )
          (v_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=512, bias=False)
          )
          (o_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=2048, bias=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=8192, bias=False)
          )
          (up_proj): LayerWithLoRA(
            (base): Linear(in_features=2048, out_features=8192, bias=False)
          )
          (down_proj): LayerWithLoRA(
 

In [10]:
# make sure only A, B parameters are trainable
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

model.layers.0.self_attn.q_proj.A torch.Size([2048, 4])
model.layers.0.self_attn.q_proj.B torch.Size([4, 2048])
model.layers.0.self_attn.k_proj.A torch.Size([2048, 4])
model.layers.0.self_attn.k_proj.B torch.Size([4, 512])
model.layers.0.self_attn.v_proj.A torch.Size([2048, 4])
model.layers.0.self_attn.v_proj.B torch.Size([4, 512])
model.layers.0.self_attn.o_proj.A torch.Size([2048, 4])
model.layers.0.self_attn.o_proj.B torch.Size([4, 2048])
model.layers.0.mlp.gate_proj.A torch.Size([2048, 4])
model.layers.0.mlp.gate_proj.B torch.Size([4, 8192])
model.layers.0.mlp.up_proj.A torch.Size([2048, 4])
model.layers.0.mlp.up_proj.B torch.Size([4, 8192])
model.layers.0.mlp.down_proj.A torch.Size([8192, 4])
model.layers.0.mlp.down_proj.B torch.Size([4, 2048])
model.layers.1.self_attn.q_proj.A torch.Size([2048, 4])
model.layers.1.self_attn.q_proj.B torch.Size([4, 2048])
model.layers.1.self_attn.k_proj.A torch.Size([2048, 4])
model.layers.1.self_attn.k_proj.B torch.Size([4, 512])
model.layers.1.se

In [11]:
# ratio of trainable / total
total_trainable_params = sum(p.numel() for p in base_model_lora.parameters() if p.requires_grad)
total_prams= sum(p.numel() for p in base_model_lora.parameters())
print(total_trainable_params/total_prams)


0.0022751285133457123


# Load dataset

In [12]:
ds = load_dataset("openai/gsm8k", "main")

In [13]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [14]:
train_ds = ds["train"]
test_ds = ds["test"]

In [15]:
print(train_ds)

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})


In [29]:
train_dataset = GSMDataset(train_ds, tokenizer)
test_dataset = GSMDataset(test_ds, tokenizer)

Max tokens: 460
Max tokens: 416


In [30]:
from torch.utils.data import random_split

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [31]:
print(len(train_loader), len(val_loader), len(test_loader))

421 47 83


# Fine-tune process

In [32]:
learning_rate = 1e-4
optimizer = torch.optim.AdamW(base_model_lora.parameters(), lr=learning_rate)
num_epochs = 3



In [33]:
train(base_model_lora, optimizer, train_loader, val_loader, num_epochs, device)

KeyboardInterrupt: 